# **EECS 6895 - Assignment 3**

This notebook contains three parts:
- **Problem 1:** CRIT Socratic Reasoning
- **Problem 2:** Big Five Personality Classification
- **Problem 3:** Short conceptual questions

**Important Notes**
- Use a GPU runtime.
- Please leave all helper functions unchanged unless the notebook asks you to modify them.
- Only modify code in the TODO sections.
- Remember to fill in all short answer questions, including in Problems 1 and 2.
- For Problem 1, please ensure your outputs match the specified JSON schema exactly.

## **0. Setup**

**Please do not change any code in this section.**

In [ ]:
# !pip -q install transformers accelerate datasets empath beautifulsoup4 lxml scikit-learn pandas numpy

In [ ]:
import re
import json
import random
import textwrap
import xml.etree.ElementTree as ElementTree

import numpy as np
import pandas as pd
import requests
import torch

from datasets import load_dataset
from empath import Empath

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.multioutput import MultiOutputClassifier
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(RANDOM_SEED)
else:
  raise Exception("Please switch to GPU runtime!")

In [ ]:

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  trust_remote_code=True,
  torch_dtype=torch.float16,
  device_map="cuda",
)
model.eval()

In [ ]:
def generate_text(user_prompt, system_prompt="You are a careful assistant that exactly follows output formats.", max_new_tokens=256):
  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
  ]
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(text, return_tensors="pt").to(model.device)
  with torch.no_grad():
    outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=False,
      pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def extract_first_json_block(text):
  text = text.strip()
  try:
    return json.loads(text)
  except Exception:
    pass
  fenced = re.search(r"```json\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.DOTALL)
  if fenced:
    return json.loads(fenced.group(1))
  for pattern in [r"(\{.*\})", r"(\[.*\])"]:
    m = re.search(pattern, text, flags=re.DOTALL)
    if m:
      return json.loads(m.group(1))
  raise ValueError("Could not parse JSON from model output. Raw output:\n" + text[:1000])

def pretty(obj):
  print(json.dumps(obj, indent=2, ensure_ascii=False))

## **1. Problem 1: CRIT Socratic Reasoning**
In this problem you will compare the following:
- **Single-shot** prompting baseline
- **Staged CRIT pipeline** with Definition, Elenchus, and Dialectic
- **Breadth-to-depth Socratic dialogue**
- **Low/high-contentiousness** rewrites

The source text is pulled directly from public  abstracts from:
- **SocraSynth: Multi-LLM Reasoning with Conditional Statistics**
- **EVINCE: Optimizing Multi-LLM Dialogues Using Conditional Statistics and Information Theory**

**Please fill out TODOs 1-9 in code and the 3 short responses about your findings.**

In [ ]:
def fetch_arxiv_metadata(arxiv_id):
  url = f"https://export.arxiv.org/api/query?id_list={arxiv_id}"
  response = requests.get(url, timeout=30)
  response.raise_for_status()
  root = ElementTree.fromstring(response.text)
  ns = {"atom": "http://www.w3.org/2005/Atom"}
  entry = root.find("atom:entry", ns)
  if entry is None:
    raise ValueError(f"No arXiv entry found for {arxiv_id}")
  title = entry.find("atom:title", ns).text.strip().replace("\n", " ")
  summary = entry.find("atom:summary", ns).text.strip().replace("\n", " ")
  return {"arxiv_id": arxiv_id, "title": title, "text": summary}

passages = [
  fetch_arxiv_metadata("2402.06634"),
  fetch_arxiv_metadata("2408.14575"),
]

for passage in passages:
  print("=" * 80)
  print(passage["title"])
  print("arXiv:", passage["arxiv_id"])
  print(textwrap.fill(passage["text"], width=100))
  print()

In [ ]:
def first_n_sentences(text, n=2):
  parts = re.split(r"(?<=[.!?])\s+", text.strip())
  return " ".join(parts[:n])

socratic_topic = first_n_sentences(passages[1]["text"], n=2)
print(socratic_topic)

### Required JSON schemas for Problem 1

**Please output your answers to the following subproblems in these formats.**

**Single-shot output**
```
{
  "main_claim": str,
  "supporting_reasons": [str, ...],   # length 2 or 3
  "evidence": [str, ...],             # same length as supporting_reasons "counterargument": str
}
```
  
**Definition output**
```
{
  "main_claim": str,
  "supporting_reasons": [str, ...]    # length 2 or 3
}
```

**Elenchus output**
```
{
  "reason": str,
  "evidence_text": str,
  "evidence_type": str,               # empirical, example, expert_opinion, conceptual_reasoning, none
  "validity_score": int,              # 1 to 5
  "credibility_score": int            # 1 to 5
}
```

**Dialectic output**
```
{
  "reason": str,
  "counterargument": str,
  "counter_strength": int             # 1 to 5
}
```

**Breadth output**
```
{
  "broad_questions": [str, str, str, str]
}
```

**Depth output**
```
{
  "selected_question": str,
  "deep_questions": [str, str, str],
  "focused_answer": str
}
```

**Contentiousness output**
```
{
  "contentiousness": str,             # "low" or "high"
  "response": str
}
```

In [ ]:
# TODO 1: Write the single-shot prompt.
# The model must return a JSON object with the exact single-shot schema.

def build_single_shot_prompt(passage):
  return f"""Read the following passage and analyze it in a single step.

Passage: {passage['text']}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "main_claim": "<the main claim or conclusion of the passage>",
  "supporting_reasons": ["<reason 1>", "<reason 2>"],
  "evidence": ["<evidence for reason 1>", "<evidence for reason 2>"],
  "counterargument": "<one counterargument to the main claim>"
}}

Rules:
- supporting_reasons must have 2 or 3 items
- evidence must have the same number of items as supporting_reasons
- Output valid JSON only"""

In [ ]:
# TODO 2: Write the Definition prompt.
# The model must return a JSON object with the exact Definition schema.

def build_definition_prompt(passage):
  return f"""Read the following passage and identify its main claim and supporting reasons.

Passage: {passage['text']}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "main_claim": "<the central thesis or conclusion of the passage>",
  "supporting_reasons": ["<reason 1>", "<reason 2>"]
}}

Rules:
- supporting_reasons must have 2 or 3 items
- Output valid JSON only"""

In [ ]:
# TODO 3: Write the Elenchus prompt.
# The model must return a JSON object with the exact Elenchus schema.

def build_elenchus_prompt(passage, reason):
  return f"""You are evaluating a supporting reason extracted from a passage.

Passage: {passage['text']}

Reason to evaluate: {reason}

Examine the evidence behind this reason and assess its quality.

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "reason": "{reason}",
  "evidence_text": "<quote or paraphrase of the evidence from the passage supporting this reason>",
  "evidence_type": "<one of: empirical, example, expert_opinion, conceptual_reasoning, none>",
  "validity_score": <integer 1 to 5>,
  "credibility_score": <integer 1 to 5>
}}

Rules:
- evidence_type must be exactly one of: empirical, example, expert_opinion, conceptual_reasoning, none
- validity_score and credibility_score must be integers between 1 and 5
- Output valid JSON only"""

In [ ]:
# TODO 4: Write the Dialectic prompt.
# The model must return a JSON object with the exact Dialectic schema.

def build_dialectic_prompt(passage, reason):
  return f"""You are constructing a counterargument to a supporting reason from a passage.

Passage: {passage['text']}

Reason: {reason}

Generate a counterargument that challenges this reason.

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "reason": "{reason}",
  "counterargument": "<a specific counterargument that challenges the reason above>",
  "counter_strength": <integer 1 to 5>
}}

Rules:
- counter_strength must be an integer between 1 (weak) and 5 (very strong)
- Output valid JSON only"""

In [ ]:
def run_single_shot(passage):
  prompt = build_single_shot_prompt(passage)
  raw = generate_text(prompt, max_new_tokens=256)
  return extract_first_json_block(raw)

def run_definition(passage):
  prompt = build_definition_prompt(passage)
  raw = generate_text(prompt, max_new_tokens=192)
  return extract_first_json_block(raw)

def run_elenchus(passage, reason):
  prompt = build_elenchus_prompt(passage, reason)
  raw = generate_text(prompt, max_new_tokens=192)
  return extract_first_json_block(raw)

def run_dialectic(passage, reason):
  prompt = build_dialectic_prompt(passage, reason)
  raw = generate_text(prompt, max_new_tokens=192)
  return extract_first_json_block(raw)

In [ ]:
# TODO 5: Implement the deterministic score aggregation function.
# Keep the final score on a 1 to 5 scale.

def compute_final_score(elenchus_outputs, dialectic_outputs):
  # Collect validity and credibility scores from elenchus
  validity_scores = [e.get("validity_score", 3) for e in elenchus_outputs]
  credibility_scores = [e.get("credibility_score", 3) for e in elenchus_outputs]
  # Collect counter_strength scores from dialectic (inverted: strong counter weakens the argument)
  counter_strengths = [d.get("counter_strength", 3) for d in dialectic_outputs]

  avg_validity = sum(validity_scores) / len(validity_scores) if validity_scores else 3
  avg_credibility = sum(credibility_scores) / len(credibility_scores) if credibility_scores else 3
  avg_counter = sum(counter_strengths) / len(counter_strengths) if counter_strengths else 3

  # Evidence quality: average of validity and credibility
  evidence_quality = (avg_validity + avg_credibility) / 2
  # Final score: evidence quality penalized by counter strength, clamped to [1, 5]
  raw_score = evidence_quality - 0.3 * (avg_counter - 3)
  return round(max(1.0, min(5.0, raw_score)), 2)

In [ ]:
# TODO 6: Build a comparison table with one row per passage.
# Include at least: title, single_shot_claim, staged_claim, num_reasons, final_score.

def build_reasoning_comparison_table(passages):
  rows = []
  for passage in passages:
    # Single-shot pipeline
    single_shot = run_single_shot(passage)

    # Staged CRIT pipeline
    definition = run_definition(passage)
    reasons = definition.get("supporting_reasons", [])

    elenchus_outputs = [run_elenchus(passage, r) for r in reasons]
    dialectic_outputs = [run_dialectic(passage, r) for r in reasons]

    final_score = compute_final_score(elenchus_outputs, dialectic_outputs)

    rows.append({
      "title": passage["title"],
      "single_shot_claim": single_shot.get("main_claim", ""),
      "staged_claim": definition.get("main_claim", ""),
      "num_reasons": len(reasons),
      "final_score": final_score,
    })

  return pd.DataFrame(rows)

In [ ]:
# TODO 7: Write the breadth prompt.
# The model must return a JSON object with the exact Breadth schema.

def build_breadth_prompt(topic_text):
  return f"""Given the following topic, generate four broad exploratory questions that cover different dimensions of the subject.

Topic: {topic_text}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "broad_questions": ["<question 1>", "<question 2>", "<question 3>", "<question 4>"]
}}

Rules:
- broad_questions must contain exactly 4 questions
- Each question should explore a different angle or dimension of the topic
- Output valid JSON only"""

In [ ]:
# TODO 8: Write the depth prompt.
# The model must return a JSON object with the exact Depth schema.

def build_depth_prompt(topic_text, selected_question):
  return f"""You are conducting a focused Socratic inquiry. Starting from a selected broad question, generate deeper follow-up questions and provide a focused answer.

Topic: {topic_text}

Selected question: {selected_question}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "selected_question": "{selected_question}",
  "deep_questions": ["<deeper follow-up question 1>", "<deeper follow-up question 2>", "<deeper follow-up question 3>"],
  "focused_answer": "<a concise focused answer to the selected question based on the topic>"
}}

Rules:
- deep_questions must contain exactly 3 questions that drill deeper into the selected question
- focused_answer should be 1-3 sentences directly addressing the selected question
- Output valid JSON only"""

In [ ]:
# TODO 9: Write the contentiousness-controlled prompt.
# The model must return a JSON object with the exact contentiousness schema.

def build_contentiousness_prompt(topic_text, contentiousness_level):
  if contentiousness_level == "low":
    style_instruction = (
      "Respond in a calm, collaborative, and neutral tone. "
      "Acknowledge multiple perspectives and use measured, diplomatic language. "
      "Avoid provocative or polarizing statements."
    )
  else:
    style_instruction = (
      "Respond in a bold, assertive, and challenging tone. "
      "Take a strong stance, use emphatic language, and highlight tensions and disagreements. "
      "Do not shy away from provocative claims."
    )

  return f"""Respond to the following topic with the specified contentiousness level.

Topic: {topic_text}

Contentiousness level: {contentiousness_level}
Style instruction: {style_instruction}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "contentiousness": "{contentiousness_level}",
  "response": "<your response to the topic in the specified tone>"
}}

Rules:
- contentiousness must be exactly "{contentiousness_level}"
- response should be 2-4 sentences
- Output valid JSON only"""

In [ ]:
# Run the single-shot and staged pipelines on the two passages.
reasoning_df = build_reasoning_comparison_table(passages)
reasoning_df

**Short Response #1**  
In 1 to 3 sentences, compare the single-shot and staged CRIT pipelines. Which one appears more structured or simpler?

**Answer: TODO**

In [ ]:
breadth_prompt = build_breadth_prompt(socratic_topic)
breadth_output = extract_first_json_block(generate_text(breadth_prompt, max_new_tokens=192))
pretty(breadth_output)

In [ ]:
FOCUS_INDEX = 1
selected_question = breadth_output["broad_questions"][FOCUS_INDEX]

depth_prompt = build_depth_prompt(socratic_topic, selected_question)
depth_output = extract_first_json_block(generate_text(depth_prompt, max_new_tokens=256))
pretty(depth_output)

**Short response #2**  
In 1 to 3 sentences, explain how the dialogue moved from broad exploration to a more focused line of reasoning.

**Answer: TODO**

In [ ]:
low_prompt = build_contentiousness_prompt(socratic_topic, "low")
high_prompt = build_contentiousness_prompt(socratic_topic, "high")

low_output = extract_first_json_block(generate_text(low_prompt, max_new_tokens=192))
high_output = extract_first_json_block(generate_text(high_prompt, max_new_tokens=192))

pretty(low_output)
print()
pretty(high_output)

**Short response #3**  
In 1 to 3 sentences, compare the low-contentiousness and high-contentiousness responses. Focus on tone, emphasis, and language.

**Answer: TODO**

## **2. Problem 2: Big Five Personality Classification from Text**

In this problem, you will build text-based models to predict the Big Five personality traits from essay data.

You will complete the following tasks:
1. Load and inspect the `essays-big5` dataset.
2. Build **TF-IDF** features from raw essay text.
3. Build **Empath** psycholinguistic category features.
4. Train three **multi-output classification** models:
   - TF-IDF only
   - Empath only
   - Combined TF-IDF and Empath
5. Perform an **ablation study** by removing one Empath category.
6. Evaluate all models using **Accuracy, F1-score, and ROC-AUC**.
7. Answer the short analysis questions at the end.

**Please fill out TODOs 10-15 in code and the 2 short responses about your findings.**

In [ ]:
# Load the public essays-big5 dataset.
dataset = load_dataset("jingjietan/essays-big5")
print(dataset)

# Build a single DataFrame, then create train/val/test partitions.
frames = []
for split_name in dataset.keys():
  df_split = dataset[split_name].to_pandas()
  df_split["source_split"] = split_name
  frames.append(df_split)

df = pd.concat(frames, ignore_index=True)
print(df.head(2))
print(df.columns.tolist())

In [ ]:
# Standardize column names if needed.
text_col_candidates = [c for c in df.columns if c.lower() in ["text", "essay", "essays"]]
if not text_col_candidates:
  raise ValueError(f"Could not find essay text column. Columns: {df.columns.tolist()}")
TEXT_COL = text_col_candidates[0]

trait_map = {}
for trait in ["O", "C", "E", "A", "N"]:
  matches = [c for c in df.columns if c.lower() == trait.lower()]
  if not matches:
    raise ValueError(f"Could not find trait column {trait}. Columns: {df.columns.tolist()}")
  trait_map[trait] = matches[0]

# Keep required columns and remove missing text.
work_df = df[[TEXT_COL] + [trait_map[t] for t in ["O", "C", "E", "A", "N"]]].copy()
work_df = work_df.rename(columns={TEXT_COL: "text", **trait_map})
work_df = work_df.dropna(subset=["text"]).reset_index(drop=True)

# Binarize trait labels.
for trait in ["O", "C", "E", "A", "N"]:
  work_df[trait] = work_df[trait].astype(int)

# Deterministic split: 70/15/15.
perm = np.random.RandomState(RANDOM_SEED).permutation(len(work_df))
work_df = work_df.iloc[perm].reset_index(drop=True)

n = len(work_df)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_df = work_df.iloc[:n_train].reset_index(drop=True)
val_df = work_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df = work_df.iloc[n_train + n_val:].reset_index(drop=True)

print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

In [ ]:
# TODO 10: Display dataset statistics and label balance information. Use the columns "split", "num_examples", "avg_chars", "{O/C/E/A/N}_positive_rate".

def summarize_split(name, split_df):
  row = {
    "split": name,
    "num_examples": len(split_df),
    "avg_chars": round(split_df["text"].str.len().mean(), 1),
  }
  for trait in ["O", "C", "E", "A", "N"]:
    row[f"{trait}_positive_rate"] = round(split_df[trait].mean(), 4)
  return row

stats_df = pd.DataFrame([
  summarize_split("train", train_df),
  summarize_split("val", val_df),
  summarize_split("test", test_df),
])

stats_df

In [ ]:
# TODO 11: Build TF-IDF features from raw essay text. Do not use precomputed features.

def build_tfidf_features(train_texts, val_texts, test_texts):
  vectorizer = TfidfVectorizer(
    max_features=10000,
    sublinear_tf=True,
    min_df=2,
    ngram_range=(1, 2),
  )
  X_train = vectorizer.fit_transform(train_texts)
  X_val = vectorizer.transform(val_texts)
  X_test = vectorizer.transform(test_texts)
  return vectorizer, X_train, X_val, X_test

vectorizer, X_train_tfidf, X_val_tfidf, X_test_tfidf = build_tfidf_features(
  train_df["text"], val_df["text"], test_df["text"]
)

print(X_train_tfidf.shape, X_val_tfidf.shape, X_test_tfidf.shape)

In [ ]:
lexicon = Empath()
EMPATH_CATEGORIES = [
  "achievement",
  "social_media",
  "friends",
  "work",
  "positive_emotion",
  "negative_emotion",
  "help",
  "communication",
  "reading",
  "thinking",
]

In [ ]:
# TODO 12: Build Empath-based category features. Use lexicon.analyze(text, categories=EMPATH_CATEGORIES, normalize=True).

def build_empath_features(texts):
  rows = []
  for text in texts:
    scores = lexicon.analyze(text, categories=EMPATH_CATEGORIES, normalize=True)
    # If analyze returns None (empty text), fill with zeros
    if scores is None:
      scores = {cat: 0.0 for cat in EMPATH_CATEGORIES}
    rows.append([scores.get(cat, 0.0) for cat in EMPATH_CATEGORIES])
  return np.array(rows, dtype=np.float32)

X_train_empath = build_empath_features(train_df["text"])
X_val_empath = build_empath_features(val_df["text"])
X_test_empath = build_empath_features(test_df["text"])

# Provided code: scale Empath feature dimensions using train statistics.
empath_scaler = StandardScaler()
X_train_empath = empath_scaler.fit_transform(X_train_empath)
X_val_empath = empath_scaler.transform(X_val_empath)
X_test_empath = empath_scaler.transform(X_test_empath)

print(X_train_empath.shape, X_val_empath.shape, X_test_empath.shape)

In [ ]:
y_train = train_df[["O", "C", "E", "A", "N"]].to_numpy(dtype=np.int64)
y_val = val_df[["O", "C", "E", "A", "N"]].to_numpy(dtype=np.int64)
y_test = test_df[["O", "C", "E", "A", "N"]].to_numpy(dtype=np.int64)

X_train_combined = hstack([X_train_tfidf, csr_matrix(X_train_empath)])
X_val_combined = hstack([X_val_tfidf, csr_matrix(X_val_empath)])
X_test_combined = hstack([X_test_tfidf, csr_matrix(X_test_empath)])

In [ ]:
# TODO 13: Train a multi-output logistic regression classifier.

def fit_multioutput_logistic(X_train, y_train):
  clf = MultiOutputClassifier(
    LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", random_state=RANDOM_SEED),
    n_jobs=-1,
  )
  clf.fit(X_train, y_train)
  return clf

tfidf_model = fit_multioutput_logistic(X_train_tfidf, y_train)
empath_model = fit_multioutput_logistic(X_train_empath, y_train)
combined_model = fit_multioutput_logistic(X_train_combined, y_train)

In [ ]:
# TODO 14: Evaluate all models using Accuracy, F1-score, and ROC-AUC. Report per-trait values and the mean across traits.

TRAITS = ["O", "C", "E", "A", "N"]

def safe_auc(y_true, y_score):
  if len(np.unique(y_true)) < 2:
    return 0.5
  return roc_auc_score(y_true, y_score)

def evaluate_model(model, X_test, y_test, model_name):
  y_pred = model.predict(X_test)
  y_prob = np.column_stack([
    est.predict_proba(X_test)[:, 1] for est in model.estimators_
  ])

  rows = []
  for i, trait in enumerate(TRAITS):
    acc = accuracy_score(y_test[:, i], y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    auc = safe_auc(y_test[:, i], y_prob[:, i])
    rows.append({
      "model": model_name,
      "trait": trait,
      "accuracy": round(acc, 4),
      "f1": round(f1, 4),
      "roc_auc": round(auc, 4),
    })

  # Mean row across all traits
  rows.append({
    "model": model_name,
    "trait": "mean",
    "accuracy": round(np.mean([r["accuracy"] for r in rows]), 4),
    "f1": round(np.mean([r["f1"] for r in rows]), 4),
    "roc_auc": round(np.mean([r["roc_auc"] for r in rows]), 4),
  })

  return pd.DataFrame(rows)

results_df = pd.concat([
  evaluate_model(tfidf_model, X_test_tfidf, y_test, "TF-IDF"),
  evaluate_model(empath_model, X_test_empath, y_test, "Empath"),
  evaluate_model(combined_model, X_test_combined, y_test, "Combined"),
], ignore_index=True)

results_df

In [ ]:
# TODO 15: Perform an ablation study by removing one Empath category feature.

ablation_rows = []
for drop_idx, drop_cat in enumerate(EMPATH_CATEGORIES):
  # Build Empath features without the dropped category
  keep_indices = [i for i, cat in enumerate(EMPATH_CATEGORIES) if cat != drop_cat]

  X_train_abl = X_train_empath[:, keep_indices]
  X_test_abl = X_test_empath[:, keep_indices]

  abl_model = fit_multioutput_logistic(X_train_abl, y_train)

  y_pred = abl_model.predict(X_test_abl)
  y_prob = np.column_stack([
    est.predict_proba(X_test_abl)[:, 1] for est in abl_model.estimators_
  ])

  mean_acc = np.mean([accuracy_score(y_test[:, i], y_pred[:, i]) for i in range(5)])
  mean_f1 = np.mean([f1_score(y_test[:, i], y_pred[:, i], zero_division=0) for i in range(5)])
  mean_auc = np.mean([safe_auc(y_test[:, i], y_prob[:, i]) for i in range(5)])

  ablation_rows.append({
    "model": f"Empath_drop_{drop_cat}",
    "trait": "mean",
    "accuracy": round(mean_acc, 4),
    "f1": round(mean_f1, 4),
    "roc_auc": round(mean_auc, 4),
  })

empath_ablation_results = pd.DataFrame(ablation_rows)
empath_ablation_results

In [ ]:
comparison_df = pd.concat([results_df, empath_ablation_results], ignore_index=True)
comparison_df

**Short Response #1**

Compare the three models. Which performs best, and what does this suggest about the relevance of TF-IDF versus psycholinguistic features?

**Answer: TODO**


**Short Response #2**

What does the ablation study suggest about feature importance?

**Answer: TODO**

## **3. Problem 3: Conceptual Questions**

Answer each question in 1 to 3 sentences.

### Q1. Socratic Reasoning
(a) Why can dialogic and multi-agent reasoning work better than monologue-style responses?  

**Answer: TODO**


(b) What are the roles of **definition**, **elenchus**, and **dialectic** in structured reasoning?

**Answer: TODO**

### Q2. Contentiousness
(a) How does contentiousness affect **tone**, **emphasis**, and **language** in generated responses?  

**Answer: TODO**

(b) How might low-contentiousness and high-contentiousness reasoning be useful in different settings?

**Answer: TODO**

### Q3. Emotion and Representation
(a) Why would we model emotion as a **spectrum** rather than as a single discrete variable?

**Answer: TODO**

(b) How is this idea related to **Plutchik’s Wheel of Emotions** or **BEAM**?

**Answer: TODO**

### Q4. Traits and Text
(a) Why are psycholinguistic or category-based features useful for predicting personality from text?  

**Answer: TODO**

(b) What are the limitations of inferring human traits from only language?

**Answer: TODO**

### Q5. Multimodal Perception
(a) Why might a **multimodal** sentiment system outperform a text-only or visual-only system?

**Answer: TODO**

(b) In class, we saw how to represents visual sentiment concepts using **adjective-noun pairs**. Why is this a better design than using nouns alone?

**Answer: TODO**